# Activity 1: HTTP and REST Fundamentals

**Module:** Week 2 Day 3
**Estimated time:** 90 to 110 minutes
**Type:** Guided notebook, read every explanation before running its cell
**Libraries:** `requests`, `httpx`, `python-dotenv`
**AI-free zone:** write the code yourself. You may call APIs, including AI APIs later today, but do not use an AI assistant to write your code.

## What you will learn

- What a request and a response actually are: method, URL, query parameters, headers, status code, body
- Why you always set a timeout and always check the status before trusting the body
- How to pass query parameters and read a paginated or searched response
- How to read a second, differently-shaped API (Open-Meteo) and pull the fields you need out of nested JSON
- How to migrate a `requests` call to `httpx` with almost no code change

## How this notebook is built

Every small piece of behavior gets its own cell, run on its own, before
anything gets combined. When a pattern is worth reusing (like a retry with
backoff), you will see it work as plain, inline code first, and only then see
it wrapped into a function. Do not skip ahead to a later cell; the point is
watching each piece get added.

## Setup

Activity 0 created the shared environment at the repo root (one `uv init` +
`uv add`) and you selected its `.venv` as this notebook's kernel, so `requests`,
`httpx`, and `python-dotenv` are already installed. If a cell reports a missing
package, add it from the repo root:

```bash
uv add requests httpx python-dotenv
```

No signup and no API key are needed for anything in this notebook. Both APIs
you will call today, GitHub's public API and Open-Meteo, are free and keyless
at this call volume. (Activity 2 does need one key, for Gemini; that setup happens there.)

## 1. The shape of a request and a response

Every time you call a REST API, you send a **request** and you get back a
**response**. Think of it like ordering at a counter and getting a receipt.

**A request has:**
- A **method** (usually `GET` to read data, `POST` to create something)
- A **URL** (which resource you want)
- Optional **query parameters** (filters or options, appended to the URL)
- Optional **headers** (metadata about the request, such as who is calling)

**A response has:**
- A **status code** (the receipt: did it work?)
- **Headers** (metadata about the response, such as the content type)
- A **body** (the actual data, usually JSON for a modern API)

We start with the public GitHub API because it needs no signup or key at light
volume. GitHub does ask every caller to send a `User-Agent` header identifying
who is calling, which is common courtesy for any API.

In [ ]:
import requests

URL = "https://api.github.com/repos/psf/requests"
HEADERS = {"User-Agent": "techcatalyst-de-2026"}

resp = requests.get(URL, headers=HEADERS, timeout=10)
print("status code:", resp.status_code)

**Expected output**: `status code: 200`. Just the receipt so far; you have
not looked at the body yet.

In [ ]:
print("content-type:", resp.headers.get("content-type"))
print("response is an object, not the data yet:", type(resp))

**Expected output**

```text
content-type: application/json; charset=utf-8
response is an object, not the data yet: <class 'requests.models.Response'>
```

`resp` is a `Response` object, not the data. The status code and headers
describe the response; the body (the actual JSON) is separate, and you have not
read it yet. `timeout=10` is not optional in real code: without it, a hung
server can freeze your program forever.

## 2. Read the receipt before you trust the body

A `200` means success. A `404` means the resource does not exist. A `429` means
you called too often. A `5xx` means the server is broken. Always check the
status before you parse the body, because a failed call can still return a body
(often an error message) that looks like it might parse.

First, see what an unhandled bad response looks like. Let this crash.

In [ ]:
bad_url = "https://api.github.com/repos/this-org-does-not-exist/this-repo-either"
resp = requests.get(bad_url, headers=HEADERS, timeout=10)
resp.raise_for_status()  # this line raises; let it, and read the traceback

**PAUSE**: read the traceback. What exception class is it? `resp.raise_for_status()`
is the one-line way to turn any 4xx or 5xx status into a loud, specific
failure instead of quietly returning garbage. Now catch it, instead of
letting it kill the whole script.

In [ ]:
try:
    resp = requests.get(bad_url, headers=HEADERS, timeout=10)
    resp.raise_for_status()
except requests.exceptions.HTTPError as exc:
    print("caught it instead of crashing the whole script:", exc)

# PAUSE: what would happen to a real pipeline if this used `except: pass`
# instead of catching HTTPError specifically and printing it?

Now the real call: parse the body only after the status check passes.

In [ ]:
resp = requests.get(URL, headers=HEADERS, timeout=10)
resp.raise_for_status()
data = resp.json()               # .json() parses the body into Python objects
print("type after .json():", type(data))

In [ ]:
print("full_name:", data["full_name"])
print("stargazers_count:", data["stargazers_count"])
print("license name (nested):", data["license"]["name"])

**Expected output shape** (star counts change over time, so your number will
differ):

```text
type after .json(): <class 'dict'>
full_name: psf/requests
stargazers_count: 52000
license name (nested): Apache License 2.0
```

`resp.json()` turns the response body into ordinary Python: a JSON object
becomes a `dict`, a JSON array becomes a `list`. Reading a nested field like
`data["license"]["name"]` is just dict-of-dicts access, the same pattern as
Day 2's nested drills.

## 3. Query parameters

Query parameters refine what you are asking for. Pass them as a dictionary with
the `params` argument and `requests` builds the URL correctly for you
(handling spaces, special characters, and the `?key=value&key2=value2` syntax),
which is safer than building the URL string by hand.

Here we use GitHub's search endpoint.

In [ ]:
search_url = "https://api.github.com/search/repositories"
params = {"q": "data engineering", "per_page": 3, "sort": "stars"}

resp = requests.get(search_url, headers=HEADERS, params=params, timeout=10)
resp.raise_for_status()
results = resp.json()

print("total_count (matches on GitHub, not just this page):", results["total_count"])

**PAUSE**: `total_count` is how many results exist across all of GitHub, not
how many came back in this response. Now look at what actually came back.

In [ ]:
print("top 3 by stars:")
for repo in results["items"]:
    print(f"  {repo['full_name']}: {repo['stargazers_count']} stars")

**Expected output shape** (exact repos and counts change over time):

```text
top 3 by stars:
  freeCodeCamp/freeCodeCamp: 410000 stars
  ...
```

`per_page` controls how many records you actually receive, separately from
`total_count`. This distinction, "how many exist" versus "how many I got
back," is exactly what pagination deals with: a real ingestion pipeline
loops, changing an `offset` or `page` parameter, until it has pulled every
record.

## 4. Headers, rate limits, and being a polite client

Two things every real API client needs to respect:

- **Rate limits**: most APIs cap how many calls you can make per minute or per
  hour. Exceed it and you get a `429 Too Many Requests`. GitHub's anonymous
  limit is low (60 calls/hour); authenticated calls get a much higher limit.
- **Backoff**: if you get a `429`, do not immediately retry. Wait, and wait
  longer on each subsequent failure (exponential backoff), so you do not make
  the problem worse.

Build this up piece by piece. First, a plain call with no protection at all.

In [ ]:
resp = requests.get(URL, headers=HEADERS, timeout=10)
print("status:", resp.status_code)

If that had come back `429`, nothing above would have noticed; the next line
would just try to use a response you should not trust yet. Add an explicit
check.

In [ ]:
if resp.status_code == 429:
    print("rate limited")
else:
    print("not rate limited, safe to continue")

Now add the smallest possible reaction to a `429`: wait a fixed amount, then
try exactly once more.

In [ ]:
resp = requests.get(URL, headers=HEADERS, timeout=10)
if resp.status_code == 429:
    print("rate limited, waiting 1s and trying once more")
    import time
    time.sleep(1)
    resp = requests.get(URL, headers=HEADERS, timeout=10)
resp.raise_for_status()
print("succeeded:", resp.status_code)

One retry with a fixed 1-second wait is fragile: a real outage needs several
tries, waiting longer each time (exponential backoff), not one fixed guess.
You would not want to copy-paste this block before every call in your
pipeline, so now that you have seen it work, wrap it in a reusable
function.

In [ ]:
import time

def polite_get(url, headers=None, params=None, max_tries=3):
    """GET with exponential backoff on 429."""
    wait = 1
    for attempt in range(1, max_tries + 1):
        resp = requests.get(url, headers=headers, params=params, timeout=10)
        if resp.status_code != 429:
            resp.raise_for_status()
            return resp
        print(f"  attempt {attempt}: rate limited, waiting {wait}s")
        time.sleep(wait)
        wait *= 2
    raise RuntimeError("gave up after repeated rate limiting")

resp = polite_get(URL, headers=HEADERS)
print("succeeded with status:", resp.status_code)

**Expected output**: since GitHub is unlikely to rate-limit a single classroom
call, this should print `succeeded with status: 200` immediately with no
retry lines. The protection only fires under real 429 pressure, but you want
it in place before you need it, not after a pipeline fails in production. You
will build this pattern again, more fully, in Activity 2 and on Day 4.

## 5. A second real API: Open-Meteo

GitHub's API is flat and needs no key. Most real-world APIs need an API key;
a few good ones do not need one at all. Open-Meteo is a free, keyless weather
API, useful both as a second real example and as a nested-JSON shape that
differs from GitHub's.

Open-Meteo needs latitude and longitude rather than a city name, and a
`current` parameter listing exactly which fields you want back.

In [ ]:
WEATHER_URL = "https://api.open-meteo.com/v1/forecast"

# Hartford, CT
params = {
    "latitude": 41.7658,
    "longitude": -72.6734,
    "current": "temperature_2m,relative_humidity_2m,weather_code",
    "temperature_unit": "fahrenheit",
}

resp = requests.get(WEATHER_URL, params=params, timeout=10)
resp.raise_for_status()
weather = resp.json()
print("full response has these top-level keys:", list(weather.keys()))

**PAUSE**: before running the next cell, guess: where do you think the
actual temperature number lives in this response?

In [ ]:
print("units:", weather["current_units"])
print("current:", weather["current"])

In [ ]:
print("temperature (F):", weather["current"]["temperature_2m"])
print("humidity (%):", weather["current"]["relative_humidity_2m"])

**Expected output shape** (real numbers depend on the weather right now):

```text
full response has these top-level keys: ['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'current_units', 'current']

units: {'time': 'iso8601', 'interval': 'seconds', 'temperature_2m': '\u00b0F', 'relative_humidity_2m': '%', 'weather_code': 'wmo code'}

current: {'time': '2026-06-29T14:00', 'interval': 900, 'temperature_2m': 71.6, 'relative_humidity_2m': 58, 'weather_code': 3}

temperature (F): 71.6
humidity (%): 58
```

Compare this shape to GitHub's: Open-Meteo nests the numbers you asked for
inside a `current` object, and separately tells you the *units* of each field
in a matching `current_units` object, so a consumer never has to guess whether
`temperature_2m` is Celsius or Fahrenheit. `weather_code` is a plain integer
(a WMO weather code), not a text description; you translate it yourself next.
There is no universal JSON shape. Reading API documentation to find where the
field you want actually lives, and how to interpret it, is a permanent skill,
not something that goes away once you have called one API.

## 6. Your turn: build a small multi-city dataset

You will use this in the DataFrame notebook (Activity 3), so keep it.
Open-Meteo needs coordinates, not names, so `CITIES` below pairs each city
with its latitude and longitude.

In [ ]:
CITIES = {
    "Hartford,US":  (41.7658, -72.6734),
    "New York,US":  (40.7128, -74.0060),
    "Boston,US":    (42.3601, -71.0589),
    "Chicago,US":   (41.8781, -87.6298),
    "Denver,US":    (39.7392, -104.9903),
    "Seattle,US":   (47.6062, -122.3321),
    "Miami,US":     (25.7617, -80.1918),
    "Phoenix,US":   (33.4484, -112.0740),
}

Open-Meteo does not send a text description for `weather_code`, only a
number. `WMO_CODES` translates it, the same dict-lookup pattern as Day 2.

In [ ]:
WMO_CODES = {
    0: "clear sky", 1: "mainly clear", 2: "partly cloudy", 3: "overcast",
    45: "fog", 48: "depositing rime fog",
    51: "light drizzle", 53: "moderate drizzle", 55: "dense drizzle",
    61: "slight rain", 63: "moderate rain", 65: "heavy rain",
    71: "slight snow", 73: "moderate snow", 75: "heavy snow",
    80: "rain showers", 81: "moderate rain showers", 82: "violent rain showers",
    95: "thunderstorm",
}

Now loop over `CITIES`, call Open-Meteo for each one, and collect the fields
you need into a list of plain dicts (not the raw nested response).

In [ ]:
import time

weather_records = []
for city, (lat, lon) in CITIES.items():
    resp = requests.get(WEATHER_URL, params={
        "latitude": lat, "longitude": lon,
        "current": "temperature_2m,relative_humidity_2m,weather_code",
        "temperature_unit": "fahrenheit",
    }, timeout=10)
    resp.raise_for_status()
    current = resp.json()["current"]
    weather_records.append({
        "city": city,
        "temp_f": current["temperature_2m"],
        "humidity": current["relative_humidity_2m"],
        "conditions": WMO_CODES.get(current["weather_code"], "unknown"),
    })
    time.sleep(0.3)

print(f"collected {len(weather_records)} city records")
for r in weather_records:
    print(" ", r)

**Expected output shape** (8 records, one per city, real numbers will vary):

```text
collected 8 city records
  {'city': 'Hartford,US', 'temp_f': 71.6, 'humidity': 58, 'conditions': 'overcast'}
  {'city': 'New York,US', 'temp_f': 74.1, 'humidity': 52, 'conditions': 'clear sky'}
  ...
```

## Success Criteria (Section 6)

- `weather_records` has exactly 8 dicts, one per city in `CITIES`.
- Each dict has the four keys `city`, `temp_f`, `humidity`, `conditions`, no
  nested structure left over.
- You called `time.sleep(0.3)` between requests instead of firing all 8 at once.

Save this notebook (or copy `weather_records` into a small JSON file with
`json.dump`) and keep it. The DataFrame notebook (Activity 3) loads exactly
this data.

## 7. `requests` versus `httpx`

`httpx` is a modern successor to `requests` with a nearly identical API.

| Feature | requests | httpx |
|---|---|---|
| Sync API | yes | yes, nearly identical |
| Async API | no | yes, with `AsyncClient` |
| HTTP/2 | no | yes, with the `h2` extra |
| Type hints | partial | full |
| Connection reuse | `Session` | `Client` |
| Recommended for new work in 2026 | maybe | yes |

First, the call you already know, in requests, as a reminder of what you are
about to reproduce.

In [ ]:
r = requests.get(URL, headers=HEADERS, timeout=10)
r.raise_for_status()
print("requests:", r.json()["full_name"])

Now the same call in httpx. Notice how little changes: the verb, `params`,
`headers`, `raise_for_status`, and `.json()` all carry over. The recommended
pattern is a `Client` used as a context manager, so connections are pooled
and closed cleanly instead of opening a new connection per call.

In [ ]:
import httpx

with httpx.Client(timeout=10) as client:
    r2 = client.get(URL, headers=HEADERS)
    r2.raise_for_status()
    print("httpx:", r2.json()["full_name"])

**Expected output**

```text
requests: psf/requests
httpx: psf/requests
```

Both lines should print the identical repository name. The migration cost from
`requests` to `httpx` is genuinely this small. The payoff is in Activity 2:
`httpx` also does async, which `requests` cannot.

## What you should notice

- The same shape repeats every time: build params, send with a timeout, check
  the status, parse JSON. This is true whether the API needs a key or not.
- The status code is the part beginners skip and engineers never skip.
- Two real APIs (GitHub, Open-Meteo) had two different JSON shapes. Reading
  the shape (from docs or from printing `.keys()`) is not optional.
- `httpx` is close enough to `requests` that switching costs almost nothing.
- The backoff logic went from "watch it happen inline" to "a reusable
  function" only after you had seen it work; that is the order to build
  things in your own code too.

## Checkpoint

Show a neighbor: your GitHub status code and one nested field, your
`weather_records` list (8 cities), and your working `httpx` call. Then move on
to Activity 2.